# **Projet InPoDa** | *Collecte, Traitement, et Analyse de données de réseaux sociaux*
InPoDa est une plateforme fictive permettant l’analyse de données de réseaux sociaux.

- ### **Importation des modules :**

In [ ]:
# Importation des modules 
import pandas as pd
import re  # librairie permettant la suppression de caractères
import json
from textblob import TextBlob  # librairie permettant l'analyse de sentiments
from textblob_fr import PatternTagger, PatternAnalyzer
import matplotlib.pyplot as plt  # librairie permettant la visualisation de données sous forme de diagrammes

## <u>**Opérations de traitement de données**</u>
- ### **Elimination des caractères spéciaux :**

In [ ]:
# Zone d'atterrissage
def zone_datterrissage(df):
    zone_datterrissage=df.copy()
    for i in range(len(zone_datterrissage["text"])):
        #print(zone_datterrissage["text"][i])
        zone_datterrissage["text"][i]=re.sub(r"[^a-zA-Z0-9-,!?+*()=.;/\s'éèçàù@#€$%:]","",zone_datterrissage["text"][i])  # suppression des caractères spéciaux non compris dans la liste
        #print(zone_datterrissage["text"][i])
    return zone_datterrissage

df=pd.read_json('versailles_tweets_100.json')  # création du dataframe associé au fichier json contenant les tweets
la_zone_datterrissage=zone_datterrissage(df)  # création de la zone d'atterrisage sans caractères spéciaux
print(la_zone_datterrissage)

- ### **Identification de l'auteur de la publication:**

In [ ]:
#fonction qui prend un tweet et renvoie son auteur
def identification_auteur(publication):
    return publication["author_id"]

#afficher tous les auteurs de toutes les publications
for i in range(len(la_zone_datterrissage)):
    print(identification_auteur(la_zone_datterrissage.iloc[i]))

#test pour 2 publications différentes
print(identification_auteur(la_zone_datterrissage.iloc[3]))
print(identification_auteur(la_zone_datterrissage.iloc[4]))

- ### **Extraction des hashtags de la publication**

In [ ]:
#fonction qui prend un tweet et renvoie ses hashtags
def extraction_hashtag(publication):
    tags=[]
    if not pd.isna(publication["entities"]):
        if "hashtags" in publication["entities"]:
            for j in range(len(publication["entities"]["hashtags"])):
                tags.append(publication["entities"]["hashtags"][j]["tag"])
            return tags,publication["entities"]["hashtags"] # lequels des 2 retourner ???
    return tags

#afficher tous les hashtags de toutes les publications
for i in range(len(la_zone_datterrissage)):  # boucle permettant de parcourir tout les hashtags de chaque tweets
        if not pd.isna(la_zone_datterrissage.iloc[i]["entities"]):
            if "hashtags" in la_zone_datterrissage.iloc[i]["entities"]:
                for j in range(len(la_zone_datterrissage.iloc[i]["entities"]["hashtags"])):
                    print(la_zone_datterrissage.iloc[i]["entities"]["hashtags"][j]["tag"])

#test pour 2 publications différentes
print(extraction_hashtag(la_zone_datterrissage.iloc[0]))
print(extraction_hashtag(la_zone_datterrissage.iloc[4]))

- ### **Extraction des utilisateurs mentionnés**

In [ ]:
#fonction qui prend un tweet et renvoie les mentions du tweet
def extraction_mention(publication):
    mentions=[]
    if not pd.isna(publication["entities"]):
        if "mentions" in publication["entities"]:
            for j in range(len(publication["entities"]["mentions"])):
                mentions.append(publication["entities"]["mentions"][j]["username"])
            return mentions,publication["entities"]["mentions"] # lequels des 2 retourner ???
    return mentions

#afficher toutes les mentions de toutes les publications
for i in range(len(la_zone_datterrissage)):  # boucle permettant de parcourir tout les hashtags de chaque tweets
        if not pd.isna(la_zone_datterrissage.iloc[i]["entities"]):
            if "mentions" in la_zone_datterrissage.iloc[i]["entities"]:
                for j in range(len(la_zone_datterrissage.iloc[i]["entities"]["mentions"])):
                    print(la_zone_datterrissage.iloc[i]["entities"]["mentions"][j]["username"])

#test pour 2 publications différentes
print(extraction_mention(la_zone_datterrissage.iloc[0]))
print(extraction_mention(la_zone_datterrissage.iloc[1]))

- ### **Analyse du sentiment de la publication**

In [ ]:
#fonction qui prend un tweet et renvoie le sentiment (posifitf/négatif)
def analyse_sentiment(publication):
    blob = TextBlob(publication["text"], pos_tagger=PatternTagger(), analyzer=PatternAnalyzer())
    if blob.sentiment[0] > 0:
        return "Positif"
    else:
        return "Negatif"

#afficher tous les sentiments de toutes les publications
for i in range(len(la_zone_datterrissage)):
    print(analyse_sentiment(la_zone_datterrissage.iloc[i]))

#test pour 2 publications différentes
print(analyse_sentiment(la_zone_datterrissage.iloc[0]))
print(analyse_sentiment(la_zone_datterrissage.iloc[3]))

- ### **Identification du/des topics de la publication**

In [ ]:
#fonction qui prend un tweet et renvoie ses topics
def identification_topics(publication):
    topics=[]
    if type(publication["context_annotations"])==float:
        return topics
    for i in range(len(publication["context_annotations"])):
        if "domain" in publication["context_annotations"][i] and "entity" in publication["context_annotations"][i]:
            topics.append((publication["context_annotations"][i]["domain"]["name"],publication["context_annotations"][i]["entity"]["name"]))
    return topics,publication["context_annotations"] # lequels des 2 retourner ???

#afficher tous les topics de toutes les publications
for i in range(len(la_zone_datterrissage)):
        if not type(la_zone_datterrissage.iloc[i]["context_annotations"])==float:
            for j in range(len(la_zone_datterrissage.iloc[i]["context_annotations"])):
                if "domain" in la_zone_datterrissage.iloc[i]["context_annotations"][j] and "entity" in la_zone_datterrissage.iloc[i]["context_annotations"][j]:
                    print(la_zone_datterrissage.iloc[i]["context_annotations"][j]["domain"]["name"])

#test pour 2 publications différentes
print(identification_topics(la_zone_datterrissage.iloc[0]))
print(identification_topics(la_zone_datterrissage.iloc[1]))

## <u>**Opérations d'analyse de données**</u>

**- Affichage Matplotlib pour tout les Tops K:**

In [ ]:
# fonction qui permet l'affichage des données en diagrammes
def top_affichage(dico,k,donnee):
    dico={str(key): value for key, value in dico.items()}  # permet de convertir tout type de clé du dictionnaire en string pour l'affichage
    plt.bar(dico.keys(),dico.values())
    plt.xlabel(str(donnee))
    plt.ylabel("Occurence")
    plt.title(f"Top {k} {donnee}")
    plt.xticks(rotation=45)
    plt.show()

- ### **Top K hashtags**

In [ ]:
#fonction qui prend k et renvoie les top k hashtags
def top_k_hashtags(k):
    dict_hashtag={}
    for i in range(len(la_zone_datterrissage)):  # boucle permettant de parcourir tout les hashtags de chaque tweets
        if not pd.isna(la_zone_datterrissage.iloc[i]["entities"]):
            if "hashtags" in la_zone_datterrissage.iloc[i]["entities"]:
                for j in range(len(la_zone_datterrissage.iloc[i]["entities"]["hashtags"])):
                    if la_zone_datterrissage.iloc[i]["entities"]["hashtags"][j]["tag"] not in dict_hashtag:  # on incrémente le nombre de hashtags des tweets
                        dict_hashtag[la_zone_datterrissage.iloc[i]["entities"]["hashtags"][j]["tag"]] = 1
                    else:
                        dict_hashtag[la_zone_datterrissage.iloc[i]["entities"]["hashtags"][j]["tag"]] += 1
    liste_hashtag=sorted(dict_hashtag.items(),key=lambda item:item[1],reverse=True)  # on tri le dictionnaire en une liste décroissante
    dict_hashtag=dict(liste_hashtag[:k])  # on met la liste en dictionnaire s'arretant au nombre de k choisi
    return dict_hashtag

#test
k=int(input("Choisissez un nombre pour les tops hashtags : "))
print(top_k_hashtags(k)) #affichage du dictionnaire des top k hashtags
top_affichage(top_k_hashtags(k),k,"Hashtags") #renvoie le diagramme matplotlib

- ### **Top K utilisateurs**

In [ ]:
#fonction qui prend k et renvoie les top k utilisateurs:
def top_k_utilisateurs(k):
    dict_utilisateurs={}
    for i in range(len(la_zone_datterrissage)):  # boucle permettant de parcourir tout les hashtags de chaque tweets
        if la_zone_datterrissage.iloc[i]["author_id"] not in dict_utilisateurs:
            dict_utilisateurs[la_zone_datterrissage.iloc[i]["author_id"]] = 1
        else:
            dict_utilisateurs[la_zone_datterrissage.iloc[i]["author_id"]] += 1
    liste_utilisateur=sorted(dict_utilisateurs.items(),key=lambda item:item[1],reverse=True)  # on tri le dictionnaire en une liste décroissante
    dict_utilisateurs=dict(liste_utilisateur[:k])  # on met la liste en dictionnaire s'arretant au nombre de k choisi
    return dict_utilisateurs

#test
k=int(input("Choisissez un nombre pour les tops utilisateurs : "))
print(top_k_utilisateurs(k)) #affichage du dictionnaire des top k utilisateurs
top_affichage(top_k_utilisateurs(k),k,"Utilisateurs") #affiche le diagramme des top k utilisateurs

- ### **Top K utilisateurs mentionnés**

In [ ]:
#fonction qui prend k et renvoie les top k utilisateurs mentionnés
def top_k_utilisateurs_mentionnes(k):
    dict_utilisateurs_mentionnes={}
    for i in range(len(la_zone_datterrissage)):  # boucle permettant de parcourir tout les hashtags de chaque tweets
        if not pd.isna(la_zone_datterrissage.iloc[i]["entities"]):
            if "mentions" in la_zone_datterrissage.iloc[i]["entities"]:
                for j in range(len(la_zone_datterrissage.iloc[i]["entities"]["mentions"])):
                    if la_zone_datterrissage.iloc[i]["entities"]["mentions"][j]["username"] not in dict_utilisateurs_mentionnes:
                        dict_utilisateurs_mentionnes[la_zone_datterrissage.iloc[i]["entities"]["mentions"][j]["username"]] = 1
                    else:
                        dict_utilisateurs_mentionnes[la_zone_datterrissage.iloc[i]["entities"]["mentions"][j]["username"]] += 1
    liste_utilisateur_mentionnes=sorted(dict_utilisateurs_mentionnes.items(),key=lambda item:item[1],reverse=True)  # on tri le dictionnaire en une liste décroissante
    dict_utilisateurs=dict(liste_utilisateur_mentionnes[:k])  # on met la liste en dictionnaire s'arretant au nombre de k choisi
    return dict_utilisateurs

#test
k=int(input("Choisissez un nombre pour les tops utilisateurs : "))
print(top_k_utilisateurs_mentionnes(k)) #affichage du dictionnaire des top k utilisateurs mentionnés
top_affichage(top_k_utilisateurs_mentionnes(k),k,"Utilisateurs mentionnés") #affichage du diagramme des top k utilisateurs mentionnés

- ### **Top K topics**

In [ ]:
#fonction qui prend et renvoie le top k topics
def top_k_topics(k):
    dict_topics={}
    for i in range(len(la_zone_datterrissage)):
        if not type(la_zone_datterrissage.iloc[i]["context_annotations"])==float:
            for j in range(len(la_zone_datterrissage.iloc[i]["context_annotations"])):
                if "domain" in la_zone_datterrissage.iloc[i]["context_annotations"][j] and "entity" in la_zone_datterrissage.iloc[i]["context_annotations"][j]:
                    if la_zone_datterrissage.iloc[i]["context_annotations"][j]["domain"]["name"] not in dict_topics:
                        dict_topics[la_zone_datterrissage.iloc[i]["context_annotations"][j]["domain"]["name"]] = 1
                    else:
                        dict_topics[la_zone_datterrissage.iloc[i]["context_annotations"][j]["domain"]["name"]] += 1
    liste_topics=sorted(dict_topics.items(),key=lambda item:item[1], reverse=True)
    dict_topics=dict(liste_topics[:k])
    return dict_topics

#test
k=int(input("Choisissez un nombre pour les tops topics : "))
print(top_k_topics(k)) #affichage du dictionnaire des top k topics
top_affichage(top_k_topics(k),k,"Topics") #affichage du diagramme des tops k topics

**- Affichage Matplotlib pour tout les Nombre de ... :**

In [ ]:
# fonction qui permet l'affichage des données en diagrammes
def nbr_affichage(dico,donnee):
    dico={str(key): value for key, value in dico.items()}  # permet de convertir tout type de clé du dictionnaire en string pour l'affichage
    plt.pie(dico.values(), labels=dico.keys())
    plt.show()

- ### **Nombre de publications par utilisateur**

In [ ]:
#fonction qui prend l'utilisateure et renvoie le nombre de tweets qu'il a fait
def nombre_publications_utilisateur(user):
    le_nombre=0
    for i in range(len(la_zone_datterrissage)):
        if la_zone_datterrissage.iloc[i]["author_id"]==user:
            le_nombre+=1
    return le_nombre

#test pour tout
dict_nbr_utilisateurs={}
for i in range(len(la_zone_datterrissage)):
    if la_zone_datterrissage.iloc[i]["author_id"] not in dict_nbr_utilisateurs:
        dict_nbr_utilisateurs[la_zone_datterrissage.iloc[i]["author_id"]] = 1
    else:
        dict_nbr_utilisateurs[la_zone_datterrissage.iloc[i]["author_id"]] += 1
liste_nbr_utilisateur=sorted(dict_nbr_utilisateurs.items(),key=lambda item:item[1],reverse=True)
dict_nbr_utilisateurs=dict(liste_nbr_utilisateur)
print(dict_nbr_utilisateurs)
nbr_affichage(dict_nbr_utilisateurs,"utilisateurs")

#test pour 2 utilisateur différent
print(nombre_publications_utilisateur(1339914264522461184))
print(nombre_publications_utilisateur(717025418))

- ### **Nombre de publications par hashtag**

In [ ]:
def nombre_publications_hashtag(hashtag):
    le_nombre=0
    for i in range(len(la_zone_datterrissage)):
        if not pd.isna(la_zone_datterrissage.iloc[i]["entities"]):
            if "hashtags" in la_zone_datterrissage.iloc[i]["entities"]:
                for j in range(len(la_zone_datterrissage.iloc[i]["entities"]["hashtags"])):
                    if la_zone_datterrissage.iloc[i]["entities"]["hashtags"][j]["tag"] == hashtag:
                        le_nombre+=1
    return le_nombre

dict_nbr_hashtags={}
for i in range(len(la_zone_datterrissage)):  # boucle permettant de parcourir tout les hashtags de chaque tweets
        if not pd.isna(la_zone_datterrissage.iloc[i]["entities"]):
            if "hashtags" in la_zone_datterrissage.iloc[i]["entities"]:
                for j in range(len(la_zone_datterrissage.iloc[i]["entities"]["hashtags"])):
                    if la_zone_datterrissage.iloc[i]["entities"]["hashtags"][j]["tag"] not in dict_nbr_hashtags:  # on incrémente le nombre de hashtags des tweets
                        dict_nbr_hashtags[la_zone_datterrissage.iloc[i]["entities"]["hashtags"][j]["tag"]] = 1
                    else:
                        dict_nbr_hashtags[la_zone_datterrissage.iloc[i]["entities"]["hashtags"][j]["tag"]] += 1
liste_nbr_hashtag=sorted(dict_nbr_hashtags.items(),key=lambda item:item[1],reverse=True)
dict_nbr_hashtags=dict(liste_nbr_hashtag)
print(dict_nbr_hashtags)
nbr_affichage(dict_nbr_hashtags,"hashtags")

print(nombre_publications_hashtag("CIV"))

- ### **Nombre de publications par topic**

In [ ]:
def nombre_publications_topic(topic):
    le_nombre=0
    for i in range(len(la_zone_datterrissage)):
        if not type(la_zone_datterrissage.iloc[i]["context_annotations"])==float:
            for j in range(len(la_zone_datterrissage.iloc[i]["context_annotations"])):
                if "domain" in la_zone_datterrissage.iloc[i]["context_annotations"][j] and "entity" in la_zone_datterrissage.iloc[i]["context_annotations"][j]:
                    if la_zone_datterrissage.iloc[i]["context_annotations"][j]["domain"]["name"]==topic:
                        le_nombre+=1
                    if la_zone_datterrissage.iloc[i]["context_annotations"][j]["entity"]["name"] == topic:
                        le_nombre+=1
    return le_nombre

dict_topics={}
for i in range(len(la_zone_datterrissage)):
        if not type(la_zone_datterrissage.iloc[i]["context_annotations"])==float:
            for j in range(len(la_zone_datterrissage.iloc[i]["context_annotations"])):
                if "domain" in la_zone_datterrissage.iloc[i]["context_annotations"][j] and "entity" in la_zone_datterrissage.iloc[i]["context_annotations"][j]:
                    if la_zone_datterrissage.iloc[i]["context_annotations"][j]["domain"]["name"] not in dict_topics:
                        dict_topics[la_zone_datterrissage.iloc[i]["context_annotations"][j]["domain"]["name"]] = 1
                    else:
                        dict_topics[la_zone_datterrissage.iloc[i]["context_annotations"][j]["domain"]["name"]] += 1
liste_topics=sorted(dict_topics.items(),key=lambda item:item[1], reverse=True)
dict_topics=dict(liste_topics)
print(dict_topics)
nbr_affichage(dict_topics,"topics")

print(nombre_publications_topic("Sports Event"))

- ### **L'ensemble de tweets d'un utilisateur spécifique**

In [ ]:
def ensemble_de_tweet_utilisateur(user):
    les_tweets=[]
    for i in range(len(la_zone_datterrissage)):
        if la_zone_datterrissage.iloc[i]["author_id"] == user:
            les_tweets.append(la_zone_datterrissage.iloc[i])
    return les_tweets

print(ensemble_de_tweet_utilisateur(1339914264522461184))

- ### **L'ensemble de tweets mentionnant un utilisateur spécifique**

In [ ]:
def ensemble_de_tweet_mentionnant_utilisateur(user):
    les_tweets=[]
    for i in range(len(la_zone_datterrissage)):
        if not pd.isna(la_zone_datterrissage.iloc[i]["entities"]):
            if "mentions" in la_zone_datterrissage.iloc[i]["entities"]:
                for j in range(len(la_zone_datterrissage.iloc[i]["entities"]["mentions"])):
                    if la_zone_datterrissage.iloc[i]["entities"]["mentions"][j]["username"] == user:
                        les_tweets.append(la_zone_datterrissage.iloc[i])
    return les_tweets

print(ensemble_de_tweet_mentionnant_utilisateur("leonna_julie"))

- ### **Les utilisateurs mentionnant un hashtag spécifique**

In [ ]:
def utilisateurs_mentionnant_hashtag(hashtag):
    les_utilisateurs=[]
    for i in range(len(la_zone_datterrissage)):
        if not pd.isna(la_zone_datterrissage.iloc[i]["entities"]):
            if "hashtags" in la_zone_datterrissage.iloc[i]["entities"]:
                for j in range(len(la_zone_datterrissage.iloc[i]["entities"]["hashtags"])):
                    if la_zone_datterrissage.iloc[i]["entities"]["hashtags"][j]["tag"] == hashtag:
                        if la_zone_datterrissage.iloc[i]["author_id"] not in les_utilisateurs:
                            les_utilisateurs.append(la_zone_datterrissage.iloc[i]["author_id"])
    return les_utilisateurs

utilisateurs_mentionnant_hashtag('CIV')

- ### **Les utilisateurs mentionnés par un utilisateur spécifique**

In [ ]:
def utilisateurs_mentionnes_utilisateur(user):
    les_utilisateurs=[]
    for i in range(len(la_zone_datterrissage)):
        if la_zone_datterrissage.iloc[i]["author_id"] == user:
            for j in range(len(la_zone_datterrissage)):
                if not pd.isna(la_zone_datterrissage.iloc[i]["entities"]):
                    if "mentions" in la_zone_datterrissage.iloc[i]["entities"]:
                        for k in range(len(la_zone_datterrissage.iloc[i]["entities"]["mentions"])):
                            if la_zone_datterrissage.iloc[i]["entities"]["mentions"][k]["username"] not in les_utilisateurs:
                                les_utilisateurs.append(la_zone_datterrissage.iloc[i]["entities"]["mentions"][k]["username"])
    return les_utilisateurs

print(utilisateurs_mentionnes_utilisateur(992904738516717568))